# Upstage Document Type Classification – Clean Baseline + Blackbar Orientation Fix

이 노트북은 아래를 포함한 **제출 가능한** 기본 파이프라인입니다.

- ✅ train/val split 누수 제거
- ✅ val 평가 로직 정상화
- ✅ confusion matrix / misclf 저장
- ✅ 블러바(검은 마스크) 기반 0/90/180/270 회전 정렬 (옵션)

> `APPLY_ORIENT_FIX=True`로 켜면 블러바 기준 자동 정렬을 적용합니다.


In [ ]:

# ===== 1) Imports / Seed / Config =====
import os, random, zipfile
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import cv2
from PIL import Image

import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm import tqdm
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import timm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ===== Paths =====
DATA_ROOT = "../data"      # baseline/ 노트북이면 보통 ../data
TRAIN_DIR = f"{DATA_ROOT}/train"
TEST_DIR  = f"{DATA_ROOT}/test"
TRAIN_CSV = f"{DATA_ROOT}/train.csv"
SAMPLE_SUB_CSV = f"{DATA_ROOT}/sample_submission.csv"

# ===== Hyperparams =====
NUM_CLASSES = 17
MODEL_NAME  = "resnet18"
IMG_SIZE    = 224
BATCH_SIZE  = 64
EPOCHS      = 10
LR          = 3e-4
WEIGHT_DECAY= 1e-2
LABEL_SMOOTH= 0.05

APPLY_ORIENT_FIX = False   # 블러바 기반 0/90/180/270 정렬
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)
print("IMG_SIZE:", IMG_SIZE, "BATCH:", BATCH_SIZE, "EPOCHS:", EPOCHS)


In [ ]:

# ===== 2) (Optional) W&B init (one-time) =====
# W&B를 안 쓰면 USE_WANDB=False로 두면 됨
USE_WANDB = False

if USE_WANDB:
    import wandb
    wandb.init(
        project="upstage-doccls",
        name=f"{MODEL_NAME}_img{IMG_SIZE}",
        config={
            "seed": SEED,
            "model": MODEL_NAME,
            "img_size": IMG_SIZE,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "lr": LR,
            "weight_decay": WEIGHT_DECAY,
            "label_smooth": LABEL_SMOOTH,
            "apply_orient_fix": APPLY_ORIENT_FIX,
        },
        reinit=False
    )


In [ ]:

# ===== 3) Train/Val Split (NO leakage) =====
df = pd.read_csv(TRAIN_CSV)
assert "target" in df.columns, "train.csv에 target 컬럼이 있어야 합니다."

id_col = df.columns[0]  # 파일명 컬럼 (예: ID)
print("ID column:", id_col)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(sss.split(df[id_col], df["target"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df   = df.iloc[val_idx].reset_index(drop=True)

TRAIN_SPLIT_CSV = f"{DATA_ROOT}/train_split.csv"
VAL_SPLIT_CSV   = f"{DATA_ROOT}/val_split.csv"
train_df.to_csv(TRAIN_SPLIT_CSV, index=False)
val_df.to_csv(VAL_SPLIT_CSV, index=False)

print("train size:", len(train_df), "val size:", len(val_df))
print("saved:", TRAIN_SPLIT_CSV, VAL_SPLIT_CSV)

print("train dist:\n", train_df["target"].value_counts().sort_index())
print("val dist:\n",   val_df["target"].value_counts().sort_index())


In [ ]:

# ===== 4) Transforms =====
trn_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(),
    ToTensorV2(),
])


## Blackbar-based orientation fix (0/90/180/270)

블러바(개인정보 마스킹)가 또렷하면, 검정 마스크로 긴 직사각형을 잡아 **방향 정렬**에 쓰기 좋습니다.

- 검정 마스크 → 컨투어 → 가장 긴 막대 1개 선택
- 막대가 세로면 90도 회전해서 가로로
- (선택) 0 vs 180 판별은 OCR confidence 같은 신호로 확장 가능


In [ ]:

def _rotate_rgb(img_rgb, angle):
    # angle in {0,90,180,270}
    if angle == 0:
        return img_rgb
    if angle == 90:
        return cv2.rotate(img_rgb, cv2.ROTATE_90_CLOCKWISE)
    if angle == 180:
        return cv2.rotate(img_rgb, cv2.ROTATE_180)
    if angle == 270:
        return cv2.rotate(img_rgb, cv2.ROTATE_90_COUNTERCLOCKWISE)
    raise ValueError("angle must be 0/90/180/270")

def find_long_black_bar_bbox(
    img_rgb,
    black_v_thresh=60,
    min_area_ratio=0.0008,
    min_aspect=4.0,
    dilate_iter=2
):
    # return: (x,y,w,h, score) or None
    h, w = img_rgb.shape[:2]
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

    # "거의 검정" 마스크
    mask = (gray < black_v_thresh).astype(np.uint8) * 255

    # 노이즈 제거 + 연결
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=dilate_iter)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    min_area = h * w * min_area_ratio
    best = None
    best_score = -1.0

    for c in contours:
        x, y, ww, hh = cv2.boundingRect(c)
        area = ww * hh
        if area < min_area:
            continue

        aspect = max(ww / (hh + 1e-6), hh / (ww + 1e-6))
        if aspect < min_aspect:
            continue

        # 너무 큰 검정영역(테두리/배경) 방지 (필요시 조정)
        if ww > 0.95 * w and hh > 0.2 * h:
            continue

        score = max(ww, hh) + area * 0.001
        if score > best_score:
            best_score = score
            best = (x, y, ww, hh, float(score))

    return best

def choose_rotation_by_blackbar(img_rgb):
    # return: (rotated_img, angle, debug)
    bbox = find_long_black_bar_bbox(img_rgb)
    debug = {"found": bbox is not None, "bbox": bbox}

    if bbox is None:
        return img_rgb, 0, debug

    x, y, ww, hh, score = bbox
    # 막대가 세로면 90도 회전 (가로로 만들기)
    angle = 90 if hh > ww else 0

    rotated = _rotate_rgb(img_rgb, angle)
    debug["angle_stage1"] = angle
    return rotated, angle, debug

# NOTE: 0 vs 180(상하) 판별까지 하고 싶으면,
# - OCR confidence가 높은 방향을 고르거나
# - 상단 텍스트 라인 점수(hough) 등으로 확장하세요.


In [ ]:

# ===== 5) Dataset / Loader =====
class ImageDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None, has_label=True, apply_orient_fix=False):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform
        self.has_label = has_label
        self.apply_orient_fix = apply_orient_fix

        self.id_col = self.df.columns[0]
        self.target_col = "target" if has_label else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = row[self.id_col]
        img_path = os.path.join(self.img_dir, fname)

        img = np.array(Image.open(img_path).convert("RGB"))

        if self.apply_orient_fix:
            img, angle, _ = choose_rotation_by_blackbar(img)

        if self.has_label:
            target = int(row[self.target_col])
        else:
            target = -1

        if self.transform is not None:
            img = self.transform(image=img)["image"]

        return img, target, fname

trn_dataset = ImageDataset(TRAIN_SPLIT_CSV, TRAIN_DIR, transform=trn_transform, has_label=True,  apply_orient_fix=APPLY_ORIENT_FIX)
val_dataset = ImageDataset(VAL_SPLIT_CSV,   TRAIN_DIR, transform=val_transform, has_label=True,  apply_orient_fix=APPLY_ORIENT_FIX)
tst_dataset = ImageDataset(SAMPLE_SUB_CSV,  TEST_DIR,  transform=val_transform, has_label=False, apply_orient_fix=APPLY_ORIENT_FIX)

trn_loader = DataLoader(trn_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
tst_loader = DataLoader(tst_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("dataset sizes:", len(trn_dataset), len(val_dataset), len(tst_dataset))


In [ ]:

# ===== 6) Model / Optim / Loss =====
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

print("model:", MODEL_NAME, "num_classes:", NUM_CLASSES)


In [ ]:

# ===== 7) Train / Val epoch functions =====
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    all_preds, all_tgts = [], []

    for x, y, _ in tqdm(loader, leave=False):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        all_preds.append(logits.argmax(1).detach().cpu().numpy())
        all_tgts.append(y.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_tgts  = np.concatenate(all_tgts)

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_tgts, all_preds)
    f1  = f1_score(all_tgts, all_preds, average="macro")
    return avg_loss, acc, f1

@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_tgts = [], []

    for x, y, _ in tqdm(loader, leave=False):
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        all_preds.append(logits.argmax(1).detach().cpu().numpy())
        all_tgts.append(y.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_tgts  = np.concatenate(all_tgts)

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_tgts, all_preds)
    f1  = f1_score(all_tgts, all_preds, average="macro")
    return avg_loss, acc, f1, all_tgts, all_preds


In [ ]:

# ===== 8) Train loop =====
best_f1 = -1.0
best_path = "best.pt"

for epoch in range(EPOCHS):
    tr_loss, tr_acc, tr_f1 = train_one_epoch(model, trn_loader, optimizer, criterion, DEVICE)
    va_loss, va_acc, va_f1, va_tgts, va_preds = eval_one_epoch(model, val_loader, criterion, DEVICE)

    print(f"[{epoch}] train loss {tr_loss:.4f} acc {tr_acc:.4f} f1 {tr_f1:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f} f1 {va_f1:.4f}")

    if USE_WANDB:
        import wandb
        wandb.log({
            "epoch": epoch,
            "train_loss": tr_loss,
            "train_acc": tr_acc,
            "train_f1": tr_f1,
            "val_loss": va_loss,
            "val_acc": va_acc,
            "val_f1": va_f1,
        })

    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save(model.state_dict(), best_path)
        print("✅ saved best:", best_path, "best_f1:", best_f1)

        if USE_WANDB:
            import wandb
            wandb.save(best_path, policy="now")


In [ ]:

# ===== 9) Confusion / Report =====
labels = list(range(NUM_CLASSES))
cm = confusion_matrix(va_tgts, va_preds, labels=labels)

pairs = []
for t in range(NUM_CLASSES):
    for p in range(NUM_CLASSES):
        if t != p:
            pairs.append((cm[t, p], t, p))
pairs.sort(reverse=True)

print("🔥 Top-20 confusions (true -> pred):")
for i, (cnt, t, p) in enumerate(pairs[:20], 1):
    print(f"{i:02d}. {t} -> {p}: {cnt}")

print("\n=== classification_report ===")
print(classification_report(va_tgts, va_preds, digits=4))


In [ ]:

# ===== 10) Save misclassified samples (val) + zip =====
SAVE_DIR = "misclf"
os.makedirs(SAVE_DIR, exist_ok=True)

MAX_SAVE = 80
FOCUS_PAIRS = [(7,3), (3,7), (14,7), (14,3), (4,7), (4,3)]
PER_PAIR_MAX = 20

@torch.no_grad()
def save_misclassified_images(model, dataset, device):
    model.eval()
    per_pair_cnt = {pair: 0 for pair in FOCUS_PAIRS}
    saved = 0

    for i in tqdm(range(len(dataset)), desc="Scanning val (misclf)"):
        x, y_true, fname = dataset[i]
        y_true = int(y_true)

        x = x.unsqueeze(0).to(device)
        logits = model(x)
        y_pred = int(logits.argmax(1).item())

        if y_pred == y_true:
            continue

        pair = (y_true, y_pred)

        if pair in per_pair_cnt:
            if per_pair_cnt[pair] >= PER_PAIR_MAX:
                continue
            per_pair_cnt[pair] += 1

        src_path = os.path.join(dataset.img_dir, fname)
        img = Image.open(src_path).convert("RGB")

        out_name = f"{saved+1:03d}_T{y_true}_P{y_pred}__{os.path.basename(fname)}"
        img.save(os.path.join(SAVE_DIR, out_name))
        saved += 1

        if saved >= MAX_SAVE:
            break

    print("✅ saved:", saved, "to", os.path.abspath(SAVE_DIR))
    print("focus pair counts:", per_pair_cnt)

# best 로드
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))

save_misclassified_images(model, val_dataset, DEVICE)

zip_path = "misclf.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(SAVE_DIR):
        for f in files:
            full = os.path.join(root, f)
            zf.write(full, arcname=os.path.relpath(full, SAVE_DIR))
print("✅ zipped:", zip_path)


In [ ]:

# ===== 11) Test inference + submission.csv (+ optional W&B artifact) =====
@torch.no_grad()
def predict_test(model, loader, device):
    model.eval()
    preds_all = []
    for x, _, _ in tqdm(loader, desc="Infer test"):
        x = x.to(device)
        logits = model(x)
        preds_all.append(logits.argmax(1).detach().cpu().numpy())
    return np.concatenate(preds_all)

if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))

test_preds = predict_test(model, tst_loader, DEVICE)

sub = pd.read_csv(SAMPLE_SUB_CSV)
sub["target"] = test_preds
sub_path = "submission.csv"
sub.to_csv(sub_path, index=False)
print("✅ saved:", sub_path)

if USE_WANDB:
    import wandb
    art = wandb.Artifact("submission", type="prediction")
    art.add_file(sub_path)
    wandb.log_artifact(art)
